In [14]:
# 请求体是客户端发送给服务器的API数据
# 响应体是API发送给客户端的数据
from typing import Annotated
from fastapi import Body,FastAPI,Path
from pydantic import BaseModel

# 创建数据类型
class Item(BaseModel):
    name:str
    description: str | None = None
    price:float
    tax:float | None = None

app = FastAPI()


@app.post("/items/")
def create_item(item:Item):
    return item


In [2]:
@app.post("/items/")
def create_item(item:Item):
    # 将对象转为python字典
    item_dic = item.model_dump()
    if item.tax is not None:
        price_with_tax = item.price + item.tax
        item_dic.update({"price_with_tax":price_with_tax})
    return item_dic

In [ ]:
# 同时声明路径参数和请求体
@app.post("/items/{item_id}")
def create_item(item_id:str,item:Item):
    # ** 是python字段解包运算符
    # **item.model_dump() 将item.model_dump()返回的字典拆开，将里面的键值对平铺在外层字典中
    # item_id 顺序会影响到最终item_id的值，假设item对象也有item_id ，item_id 在前会被item对象中的item_id 覆盖
    return {"item_id":item_id,**item.model_dump()}

In [6]:
# 同时声明 请求体、路径参数、查询参数
@app.post("/items/{item_id}")
def create_item(item:Item,item_id:str,q:str  | None = None):
    result = {"item_id":item_id,**item.model_dump()}
    if q:
        result.update({"q":q})
    return result

In [ ]:
# 请求体-多个参数

class User(BaseModel):
    username:str
    full_name: str | None = None


@app.put("/items/{item_id}")
def update_item(item_id:int ,item:Item,user:User,importance:Annotated[int,Body()]
                ):
    result =  {"item_id":item_id,"item":item,"user":user,"importance":importance}
    return result

In [ ]:
@app.put("/items/{item_id}")
# Annotated[int,Path 这一堆是为了显示声明item_id 是路径参数和验证合法性
def update_item(item_id:Annotated[int,Path(title="The ID of the item to get",ge=0,le=1000)],
                q:str|None=None,
                item:Item | None= None
                ):
    results = {"item_id":item_id}
    if q:
        results.update({"q":q})

    if item:
        results.update({"item":item})
    return results

In [16]:
# * 栅栏，分割符，* 右边必须按照顺序，键=值，左边不用

@app.put("/items/{item_id}")
def update_item(
    *,
    item_id:int,
    item:Item,
    user:User,
    importance:Annotated[int,Body(gt=0)],
    q: str | None = None
):
    results = {"item_id":item_id,"item":item,"user":user,"importance":importance}
    if q:
        results.update({"q":q})
    return results